In [ ]:
import pandas as pd
import numpy as np

# Load in R-exported CSV with correct CBSA joins
df = pd.read_csv("/Users/brandonsmith/DATA-510-CAPSTONE--2/DATA-510-CAPSTONE--2/data/final_data/price_changes_with_collapse_flags.csv")

# replace dots in column names with underscores for easier acess
df.columns = df.columns.str.replace('.', '_', regex = False)

print(df.shape)
print(df.dtypes[['cbsa', 'metro_name_x', 'year', 'qtr']])

(83640, 26)
cbsa            float64
metro_name_x        str
year            float64
qtr             float64
dtype: object


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_25191/3221209643.py:4: DtypeWarning: Columns (0: metro_name.x, 1: RegionName.y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/brandonsmith/DATA-510-CAPSTONE--2/DATA-510-CAPSTONE--2/data/final_data/price_changes_with_collapse_flags.csv")


In [ ]:
# sanity check: confirming Austin (12420), Boise (14260), and Tampa (45294) all have data
training_cbsa_map = {
    'Austin': 12420,
    'Boise': 14260,
    'Tampa': 45294
}

for city, code in training_cbsa_map.items():
    sub = df[df['cbsa'] == code]
    print(f"{city} ({code}): rows={len(sub)},"
          f"zhvi_yoy NA={sub['zhvi_yoy'].isna().sum()}, " # checks for NA gaps in zhvi_yoy, median_income, and price_to_income_ratio
          f"median_income NA={sub['median_income'].isna().sum()}, "
          f"price_to_income_ratio NA={sub['price_to_income_ratio'].isna().sum()}")
    
# this helps catches silent join failures before they cause problem in feature engineering

Austin (12420): rows=204,zhvi_yoy NA=204, median_income NA=144, price_to_income_ratio NA=144
Boise (14260): rows=204,zhvi_yoy NA=108, median_income NA=144, price_to_income_ratio NA=144
Tampa (45294): rows=204,zhvi_yoy NA=204, median_income NA=144, price_to_income_ratio NA=144


In [ ]:
df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) -1)
df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0 # creating is_unaffordable flag based when price_to_income ratio exceeds 5.0 (Demographia "severly unaffordable" line)
df['collapse_onset'] = df['is_unaffordable'] & ~g['is_unaffordable'].shift(1).fillna(False) # uses groupby('cbsa) so each city's history is tracked separetely, not limited mixed together

first_collapse = (
    df[df['collapse_onset']].groupby('cbsa')[['metro_name_x', 'year', 'qtr']].first()
)

print(first_collapse.loc[first_collapse.index.isin(training_cbsa_map)])

# filter out the ~140 structurally empty rows (pre-2010 data with no income match)
for city, code in training_cbsa_map.items():
    sub = df[df['cbsa'] == code].sort_values(['year', 'qtr'])
    print(f"\n{city} ({code}):")
    print(sub['price_to_income_ratio'].describe())
    print("Max ratio:", sub['price_to_income_ratio'].max())
    print("Any non-null values:", sub['price_to_income_ratio'].notna().sum())

Empty DataFrame
Columns: [metro_name_x, year, qtr]
Index: []

Austin (12420):
count    60.000000
mean      4.326560
std       0.730578
min       3.326301
25%       3.741221
50%       4.302468
75%       4.613371
max       6.130334
Name: price_to_income_ratio, dtype: float64
Max ratio: 6.130334310962185
Any non-null values: 60

Boise (14260):
count    60.000000
mean      4.516931
std       1.284007
min       2.665218
25%       3.543262
50%       4.383438
75%       5.654204
max       7.192370
Name: price_to_income_ratio, dtype: float64
Max ratio: 7.192370410987232
Any non-null values: 60

Tampa (45294):
count    60.000000
mean      3.932904
std       0.953983
min       2.537116
25%       3.072381
50%       3.951967
75%       4.526633
max       5.771200
Name: price_to_income_ratio, dtype: float64
Max ratio: 5.771200004400776
Any non-null values: 60


In [ ]:
# print only the 60 real quarters per city so onset dates are visible without truncication 
for city, code in training_cbsa_map.items():
    sub = df[df['cbsa'] == code].sort_values(['year', 'qtr']).copy()
    sub = sub[sub['price_to_income_ratio'].notna()]
    sub['is_unaffordable'] = sub['price_to_income_ratio'] > 5.0
    sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
    sub['collapse_onset_test'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']

    onset_row = sub[sub['collapse_onset_test']].head(1)
    print(f"{city}: first onset = {onset_row[['year','qtr','price_to_income_ratio']].values}")

Austin: first onset = [[2.02100000e+03 2.00000000e+00 5.19478076e+00]]
Boise: first onset = [[2019.            3.            5.04682862]]
Tampa: first onset = [[2021.            4.            5.21363528]]


In [ ]:
# testing to see how collapse onset date shifts if the threshold is 4.0, 4.5, 5.0 or 5.5
# Goal: confirm that 5.0 isn't an arbitry choise -- checck if a different threshold changes results signifcantly
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code in training_cbsa_map.items():
        sub = df[df['cbsa'] == code].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Results: 5.0 gives the tightest, most realistic cluster of onset dates for all three training cities


--- Threshold: 4.0 ---
  Austin: 2014.0Q3.0
  Boise: 2016.0Q2.0
  Tampa: 2017.0Q4.0

--- Threshold: 4.5 ---
  Austin: 2020.0Q4.0
  Boise: 2017.0Q4.0
  Tampa: 2021.0Q2.0

--- Threshold: 5.0 ---
  Austin: 2021.0Q2.0
  Boise: 2019.0Q3.0
  Tampa: 2021.0Q4.0

--- Threshold: 5.5 ---
  Austin: 2021.0Q3.0
  Boise: 2020.0Q4.0
  Tampa: 2022.0Q2.0


In [ ]:
df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0 # lock in validated 5.0 threshold across the entire dataframe (not just training cities)
g = df.groupby('cbsa')
df['collapse_onset'] = df['is_unaffordable'] & ~g['is_unaffordable'].shift(1).fillna(False)

df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) - 1) * 100 # hpi_3yr_chg as a second momentum signal alongside price_to_income_ratio

first_collapse = (
    df[df['collapse_onset']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
) # final check: confirm onset dates still match austin 2021q2, boise 2019q3, Tampa, 2021q4
print(first_collapse.loc[first_collapse.index.isin(training_cbsa_map.values())])

                             metro_name_x    year  qtr  price_to_income_ratio
cbsa                                                                         
12420.0  Austin-Round Rock-San Marcos, TX  2021.0  2.0               5.194781
14260.0                    Boise City, ID  2019.0  3.0               5.046829
45294.0                  Tampa, FL (MSAD)  2021.0  4.0               5.213635


# Feature Enginnering
Every feature below is lagged by 4 quarters (1 year) before any rolling calcuation. This ensures no feature accidentally "sees" the same-quarter data used to build collapse_onset. 

# Rules
* never let a feature use current or future infro that overlaps with label construction

In [39]:
LAG = 4

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

# Feature 1: price-to-income 5-year change
df['price_to_income_lag'] = g['price_to_income_ratio'].shift(LAG)
df['price_to_income_5yr_chg'] = df.groupby('cbsa')['price_to_income_lag'].transform(lambda x: x- x.shift(20))

# Feature 2: affordability momentum -- 3yr rolling slope of LAGGED zhvi_yoy
df['zhvi_qoq_lag'] = df.groupby('cbsa')['zhvi_qoq'].shift(LAG)

# Feature 3: hpi-based momentum (yoy change in index_sa)
df['hpi_yoy'] = df.groupby('cbsa')['index_sa'].transform(lambda x: x.pct_change(4))
df['hpi_yoy_lag'] = df.groupby('cbsa')['hpi_yoy'].shift(LAG)

# Feature 4: population acceleration, lagged
df['pop_velocity_lag'] = df.groupby('cbsa')['pop_velocity_3yr'].shift(LAG)

# Feature 5: HPI-based momentum (YoY change in index_sa), lagged
df['pop_acceleration_lag'] = df.groupby('cbsa')['pop_accleration'].shift(LAG)

print("df length after all features:", len(df))


df length after all features: 83640


In [41]:
# verification test
feature_cols = ['price_to_income_5yr_chg', 'affordability_momentum',
                 'pop_velocity_lag', 'pop_acceleration_lag', 'hpi_yoy_lag']

for col in feature_cols:
    n_rows = df[col].notna().sum()
    n_cities = df[df[col].notna()]['cbsa'].nunique()
    print(f"{col}: {n_rows} rows, {n_cities} cities")

price_to_income_5yr_chg: 120 rows, 3 cities
affordability_momentum: 16779 rows, 228 cities
pop_velocity_lag: 17304 rows, 373 cities
pop_acceleration_lag: 18796 rows, 373 cities
hpi_yoy_lag: 13200 rows, 100 cities


In [42]:
# This is not usable for holdout scoring -- only Austin/Boise/Tampa have this data
# Keep it seperated labled so it doesn't accidently get used as a modeling feature later

training_diagnostic_cols = ['price_to_income_ratio', 'price_to_income_5yr_chg']

for col in training_diagnostic_cols:
    n_cities = df[df[col].notna()]['cbsa'].nunique()
    print(f"{col}: (Training-city diagnostic only): {n_cities} cities")

price_to_income_ratio: (Training-city diagnostic only): 3 cities
price_to_income_5yr_chg: (Training-city diagnostic only): 3 cities
